# Notebook 03b: Vehicle OSNet Resume Training
Resume OSNet-x1.0 training on VeRi-776 from the checkpoint saved by Notebook 03.

**Kernel source**: `mrkdagods/03-vehicle-reid-training` (provides the checkpoint)
**Runtime**: GPU T4x2 | **Duration**: ~5-7 hours (remaining epochs only)

In [ ]:
!pip install -q torchreid

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import shutil
import glob
import re
import os.path as osp
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/reid_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_EPOCH = 150

CONFIG = {
    "dataset": "veri776",
    "veri776_root": "/kaggle/working/data",
    "batch_size_train": 64,
    "batch_size_test": 128,
    "num_workers": 2,
    "height": 224,
    "width": 224,
}

print(f"Device: {DEVICE}")
print(f"Target: {MAX_EPOCH} total epochs")

## 2. Find checkpoint from Notebook 03

In [ ]:
# nb03 output is mounted at /kaggle/input/03-vehicle-reid-training/
NB03_OUTPUT = Path("/kaggle/input/03-vehicle-reid-training")

print(f"nb03 output exists: {NB03_OUTPUT.exists()}")
if NB03_OUTPUT.exists():
    print("Contents:")
    for p in sorted(NB03_OUTPUT.rglob("*")):
        if p.is_file():
            size_mb = p.stat().st_size / 1024**2
            print(f"  {p.relative_to(NB03_OUTPUT)} ({size_mb:.1f} MB)")

# Find the latest OSNet checkpoint
osnet_model_dir = NB03_OUTPUT / "reid_models" / "vehicle_osnet_x1_0" / "model"
checkpoint_path = None
start_epoch = 0

if osnet_model_dir.exists():
    # Look for model.pth.tar-{epoch} files
    checkpoints = sorted(osnet_model_dir.glob("model.pth.tar-*"))
    if checkpoints:
        # Pick the latest epoch
        latest = checkpoints[-1]
        epoch_num = int(latest.name.split("-")[-1])
        checkpoint_path = latest
        start_epoch = epoch_num
        print(f"\nFound checkpoint: {latest.name} (epoch {epoch_num})")
    
    # Also check for model-best.pth.tar
    best = osnet_model_dir / "model-best.pth.tar"
    if best.exists():
        print(f"Best model also available: {best}")
else:
    print(f"\nWARNING: {osnet_model_dir} not found")
    print("Will train from scratch with pretrained ImageNet weights.")

print(f"\nResume from epoch: {start_epoch}")
print(f"Remaining epochs: {MAX_EPOCH - start_epoch}")

## 3. Dataset Setup

In [ ]:
import torchreid

# ============================================================
# Step 1: Prepare data directory (Kaggle /input is read-only)
# ============================================================
DATA_ROOT = Path("/kaggle/working/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

possible_roots = [
    Path("/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset"),
    Path("/kaggle/input/veri-vehicle-re-identification-dataset"),
]

input_dir = None
for root in possible_roots:
    if root.exists():
        input_dir = root
        break

if input_dir is None:
    import subprocess
    result = subprocess.run(["find", "/kaggle/input", "-maxdepth", "5", "-type", "d", "-name", "image_train"],
                            capture_output=True, text=True, timeout=30)
    print(f"Search results:\n{result.stdout}")
    raise RuntimeError(
        f"VeRi dataset not found. Tried: {[str(p) for p in possible_roots]}. "
        "Make sure 'abhyudaya12/veri-vehicle-re-identification-dataset' is attached."
    )

print(f"Dataset found at: {input_dir}")

# Find image_train directory
veri_data = None
for p in input_dir.rglob("image_train"):
    if p.is_dir():
        veri_data = p.parent
        break

if veri_data is None:
    raise RuntimeError(f"Could not find 'image_train' inside {input_dir}")

print(f"VeRi data at: {veri_data}")

symlink = DATA_ROOT / "veri"
if symlink.exists() or symlink.is_symlink():
    symlink.unlink()
symlink.symlink_to(veri_data)

for subdir in ["image_train", "image_query", "image_test"]:
    d = symlink / subdir
    if d.exists():
        n = len(list(d.iterdir()))
        print(f"  {subdir}: {n} files")

# ============================================================
# Step 2: Register custom VeRi dataset
# ============================================================
class VeRi776(torchreid.data.ImageDataset):
    dataset_dir = 'veri'

    def __init__(self, root='', **kwargs):
        self.root = osp.abspath(osp.expanduser(root))
        self.dataset_dir = osp.join(self.root, self.dataset_dir)
        train_dir = osp.join(self.dataset_dir, 'image_train')
        query_dir = osp.join(self.dataset_dir, 'image_query')
        test_dir = osp.join(self.dataset_dir, 'image_test')
        train = self._process_dir(train_dir, relabel=True)
        query = self._process_dir(query_dir, relabel=False)
        gallery = self._process_dir(test_dir, relabel=False)
        super().__init__(train, query, gallery, **kwargs)

    def _process_dir(self, dir_path, relabel=False):
        img_paths = glob.glob(osp.join(dir_path, '*.jpg'))
        if not img_paths:
            raise RuntimeError(f"No .jpg images in {dir_path}")
        pattern = re.compile(r'^(\d+)_c(\d+)')
        data = []
        for img_path in sorted(img_paths):
            fname = osp.basename(img_path)
            match = pattern.match(fname)
            if match:
                pid = int(match.group(1))
                camid = int(match.group(2)) - 1
                data.append((img_path, pid, camid))
        if not data:
            raise RuntimeError(f"No files matched VeRi pattern in {dir_path}")
        if relabel:
            pid_set = sorted(set(d[1] for d in data))
            pid2label = {pid: label for label, pid in enumerate(pid_set)}
            data = [(img_path, pid2label[pid], camid) for img_path, pid, camid in data]
        n_ids = len(set(d[1] for d in data))
        print(f"  Loaded {osp.basename(dir_path)}: {len(data)} images, {n_ids} IDs (relabel={relabel})")
        return data

torchreid.data.register_image_dataset('veri', VeRi776)
print("\nRegistered custom VeRi-776 dataset")

# ============================================================
# Step 3: Create datamanager
# ============================================================
datamanager = torchreid.data.ImageDataManager(
    root=CONFIG["veri776_root"],
    sources=["veri"],
    targets=["veri"],
    height=CONFIG["height"],
    width=CONFIG["width"],
    batch_size_train=CONFIG["batch_size_train"],
    batch_size_test=CONFIG["batch_size_test"],
    transforms=["random_flip", "random_crop", "random_erasing", "color_jitter"],
    workers=CONFIG["num_workers"],
)

print(f"\nTrain: {datamanager.num_train_pids} vehicle IDs, {datamanager.num_train_cams} cameras")

## 4. Build OSNet + Load Checkpoint

In [ ]:
# Build model (without DataParallel first, so we can load weights cleanly)
model_osnet = torchreid.models.build_model(
    name="osnet_x1_0",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=(checkpoint_path is None),  # only use ImageNet weights if no checkpoint
)
model_osnet = model_osnet.to(DEVICE)

# Load checkpoint weights (handles module. prefix from DataParallel)
if checkpoint_path is not None:
    print(f"Loading checkpoint: {checkpoint_path}")
    torchreid.utils.load_pretrained_weights(model_osnet, str(checkpoint_path))
    print(f"Loaded weights from epoch {start_epoch}")
else:
    print("No checkpoint found — training from scratch with ImageNet pretrained weights")

# Now wrap in DataParallel for multi-GPU
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model_osnet = nn.DataParallel(model_osnet)

# Build optimizer (fresh — Adam adapts quickly)
optimizer_osnet = torchreid.optim.build_optimizer(
    model_osnet, optim="adam", lr=3.5e-4, weight_decay=5e-4,
)

# Build scheduler for full run, then advance to correct position
scheduler_osnet = torchreid.optim.build_lr_scheduler(
    optimizer_osnet, lr_scheduler="cosine", max_epoch=MAX_EPOCH,
)
for _ in range(start_epoch):
    scheduler_osnet.step()

print(f"\nParameters: {sum(p.numel() for p in model_osnet.parameters()):,}")
print(f"Scheduler advanced to epoch {start_epoch}")
print(f"Current LR: {scheduler_osnet.get_last_lr()[0]:.6f}")

## 5. Resume Training

In [ ]:
engine_osnet = torchreid.engine.ImageSoftmaxEngine(
    datamanager, model_osnet,
    optimizer=optimizer_osnet, scheduler=scheduler_osnet,
    label_smooth=True,
)

print("=" * 60)
print(f"Training OSNet-x1.0 on VeRi-776 (epoch {start_epoch} -> {MAX_EPOCH})")
print("=" * 60)

train_start = time.time()
engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_osnet_x1_0"),
    max_epoch=MAX_EPOCH, start_epoch=start_epoch,
    eval_freq=20, print_freq=50,
    test_only=False,
)
osnet_time = time.time() - train_start
print(f"\nCompleted in {osnet_time / 3600:.2f} hours")

## 6. Final Evaluation

In [ ]:
engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_osnet_x1_0"),
    max_epoch=MAX_EPOCH, test_only=True,
)
print("\nOSNet-x1.0 (Vehicle) evaluation complete. See mAP and Rank-1 metrics above.")

## 7. Export Model

In [ ]:
export_dir = Path("/kaggle/working/exported_models")
export_dir.mkdir(parents=True, exist_ok=True)

# Export OSNet
src_best = OUTPUT_DIR / "vehicle_osnet_x1_0" / "model" / "model-best.pth.tar"
src_epoch = OUTPUT_DIR / "vehicle_osnet_x1_0" / "model" / f"model.pth.tar-{MAX_EPOCH}"

src = src_best if src_best.exists() else src_epoch
if src.exists():
    dst = export_dir / "vehicle_osnet_x1_0_veri776.pth"
    shutil.copy2(src, dst)
    print(f"Exported: {dst.name} ({dst.stat().st_size / 1024**2:.1f} MB)")
else:
    print(f"Warning: no model found at {src_best} or {src_epoch}")

# Also copy ResNet from nb03 output if available
resnet_src = NB03_OUTPUT / "reid_models" / "vehicle_resnet50_ibn_a" / "model" / "model-best.pth.tar"
if not resnet_src.exists():
    # Try epoch-specific checkpoint
    candidates = sorted((NB03_OUTPUT / "reid_models" / "vehicle_resnet50_ibn_a" / "model").glob("model.pth.tar-*"))
    if candidates:
        resnet_src = candidates[-1]

if resnet_src.exists():
    dst = export_dir / "vehicle_resnet50_ibn_a_veri776.pth"
    shutil.copy2(resnet_src, dst)
    print(f"Copied ResNet from nb03: {dst.name} ({dst.stat().st_size / 1024**2:.1f} MB)")
else:
    print("ResNet model not found in nb03 output — download from nb03 separately")

# Metadata
metadata = {
    "task": "vehicle_reid",
    "dataset": "veri776",
    "resumed_from_epoch": start_epoch,
    "final_epoch": MAX_EPOCH,
    "train_time_hours": osnet_time / 3600,
    "models": {
        "osnet_x1_0": {
            "architecture": "osnet_x1_0",
            "embedding_dim": 512,
            "input_size": [224, 224],
            "file": "vehicle_osnet_x1_0_veri776.pth",
        },
    },
    "trained_at": datetime.now().isoformat(),
}

with open(export_dir / "vehicle_osnet_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nExported to: {export_dir}")
print("\nAll files:")
for f in sorted(export_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024**2:.1f} MB)")